# `InMemoryDocumentIndex: DocumentIndex`

In-memory beta document index backed by a dictionary.

Marked with:

```python
@beta(message="Introduced in version 0.2.29. Underlying abstraction subject to change.")
```

It supports ID-based upserts, deletion, retrieval, and simple text search based on how many times the exact query occurs in each document.

## Fields

```python
store: dict[str, Document] = Field(default_factory=dict) # Documents stored by ID
top_k: int = 4 # Maximum number of documents returned by a search
```

## Methods

### `upsert`

Adds new documents or replaces existing documents with the same IDs.

```python
@override
upsert(
    self,
    items: Sequence[Document], # Documents to add or replace
    /,
    **kwargs: Any, # Additional arguments accepted but not used
) -> UpsertResponse # IDs successfully upserted and an empty failure list
```

A document with an existing ID replaces the value stored under that ID.

When a document has no ID, the method generates a UUID4 string, creates a model copy, assigns the generated ID to the copy, and stores it. The original document remains unchanged.

The returned `succeeded` list follows the input-document order. This implementation does not report upsert failures, so `failed` is always empty.

### `delete`

Deletes documents by ID.

```python
@override
delete(
    self,
    ids: list[str] | None = None, # Document IDs to delete
    **kwargs: Any, # Additional arguments accepted but not used
) -> DeleteResponse # Details of documents actually deleted
```

Missing IDs are ignored. The response contains only IDs that existed and were deleted, in request order:

```python
{
    "succeeded": ok_ids,
    "num_deleted": len(ok_ids),
    "num_failed": 0,
    "failed": [],
}
```

Raises `ValueError` when `ids` is `None`.

### `get`

Retrieves stored documents by ID.

```python
@override
get(
    self,
    ids: Sequence[str], # Document IDs to retrieve
    /,
    **kwargs: Any, # Additional arguments accepted but not used
) -> list[Document] # Existing documents in requested-ID order
```

Missing IDs are skipped without raising an exception. Returned documents are the objects held in `store`, not copies.

## Behaviour

The inherited retriever interfaces use this class's search implementation. A search counts occurrences using:

```python
document.page_content.count(query)
```

Results are sorted by descending occurrence count and limited to `top_k`. Matching is exact and case-sensitive, and `str.count()` counts non-overlapping occurrences. Documents with zero occurrences can still appear when fewer than `top_k` documents have higher counts.

Search results are returned as model copies rather than the objects held in `store`. Equal-count documents preserve their storage insertion order.

The inherited `aupsert()`, `adelete()`, and `aget()` methods run the corresponding synchronous methods through an executor.

In [ ]:
from langchain_core.documents import Document # Import the Document class
from langchain_core.indexing.in_memory import InMemoryDocumentIndex # Import the in-memory index

index = InMemoryDocumentIndex(top_k=2) # Return at most two search results

documents = [ # Create documents to store
    Document(id="doc1", page_content="Python is simple. Python is popular."), # Document with an ID
    Document(id="doc2", page_content="LangChain helps build AI applications."), # Another document with an ID
    Document(page_content="Python is widely used in AI."), # Document without an ID
] # Finish the document list

upsert_result = index.upsert(documents) # Add the documents to the index

print("Upsert result:", upsert_result) # Display successful and failed IDs
print("Stored IDs:", list(index.store)) # Display all stored IDs
print("Original third ID:", documents[2].id) # Original document remains unchanged

retrieved = index.get(["doc1", "missing", "doc2"]) # Retrieve documents by ID

print("\nRetrieved documents:") # Display a heading
for document in retrieved: # Visit each retrieved document
    print(document.id, ":", document.page_content) # Display its ID and content

search_results = index.invoke("Python") # Search using exact case-sensitive matching

print("\nSearch results:") # Display a heading
for document in search_results: # Visit each search result
    print(document.id, ":", document.page_content) # Display the matching document

replacement = Document( # Create a replacement document
    id="doc1",
    page_content="Python has been updated.",
)

index.upsert([replacement]) # Replace the existing document with the same ID

print("\nUpdated doc1:", index.get(["doc1"])[0].page_content) # Display updated content

delete_result = index.delete(["doc2", "missing"]) # Delete an existing and missing ID

print("Delete result:", delete_result) # Display deletion details
print("Remaining IDs:", list(index.store)) # Display remaining IDs

async_documents = await index.aget(["doc1"]) # Retrieve asynchronously in Jupyter
print("Async get:", async_documents[0].page_content) # Display asynchronous result